# Foldseek Toolkit

![Foldseek](https://proto-bio.github.io/proto-assets/images/tool/foldseek/hero.png)

This notebook demonstrates the five Foldseek tools:

1. **`foldseek-search`** — single-chain structural homology search (remote default)
2. **`foldseek-cluster`** — cluster a set of structures by structural similarity (local-only)
3. **`foldseek-multimer-search`** — multimer (complex) structural search (remote default)
4. **`foldseek-multimercluster`** — cluster a set of multi-chain assemblies by complex-level structural similarity (local-only)
5. **`foldseek-rbh`** — reciprocal-best-hits structural search between a query and a target DB (local-only)

References:
- van Kempen et al., *Nature Biotechnology* 2024 ([DOI](https://doi.org/10.1038/s41587-023-01773-0)) — Foldseek
- Kim et al., *Nature Methods* 2025 ([DOI](https://doi.org/10.1038/s41592-025-02593-7)) — Foldseek-Multimer

In [1]:
from proto_tools import Structure
from proto_tools.tools.database_retrieval import (
    AlphaFoldDBFetchConfig, AlphaFoldDBFetchInput, run_alphafold_db_fetch,
)
from proto_tools.tools.structure_alignment import (
    FoldseekClusterConfig, FoldseekClusterInput, run_foldseek_cluster,
    FoldseekMultimerClusterConfig, FoldseekMultimerClusterInput, run_foldseek_multimercluster,
    FoldseekMultimerSearchConfig, FoldseekMultimerSearchInput, run_foldseek_multimer_search,
    FoldseekRBHConfig, FoldseekRBHInput, run_foldseek_rbh,
    FoldseekSearchConfig, FoldseekSearchInput, run_foldseek_search,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API

In [2]:
display_api_reference("foldseek", "input", "run_foldseek_search")

**Input** — `FoldseekSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structure</code> | <code>Structure</code> | required | Query structure (Structure object, file path, or raw PDB/CIF string) |

## Config API

In [3]:
display_api_reference("foldseek", "config", "run_foldseek_search")

**Config** — `FoldseekSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>search_mode</code> | <code>Literal['remote', 'local']</code> | <code>'remote'</code> | 'remote' (search.foldseek.com) or 'local' (local Foldseek CLI) |
| <code>databases</code> | <code>list[Literal['pdb100', 'afdb50', 'afdb-swissprot', 'afdb-proteome', 'mgnify_esm30', 'gmgcl_id', 'BFVD', 'cath50', 'bfmd']]</code> | <code>['pdb100', 'afdb50']</code> | Remote-only — server-hosted reference databases to search |
| <code>mode</code> | <code>Literal['3diaa', 'tmalign', 'lolalign']</code> | <code>'3diaa'</code> | Remote-only — '3diaa' (fast 3Di+AA local), 'tmalign' (global), or 'lolalign' (local, slow) |
| <code>poll_interval_seconds</code> | <code>float</code> | <code>5.0</code> | Remote-only — delay between status polls |
| <code>timeout_seconds</code> | <code>float</code> | <code>600.0</code> | Remote-only — max wall-clock time for the search |
| <code>local_db</code> | <code>str &#124; None</code> | <code>None</code> | Path to a local Foldseek DB or a directory of PDB files (required for local mode) |
| <code>evalue</code> | <code>float</code> | <code>10.0</code> | E-value cutoff. Lower = stricter (1e-3 for confident homologs; default 10.0 reports all) |
| <code>sensitivity</code> | <code>float</code> | <code>9.5</code> | Prefilter sensitivity (1.0-9.5). Lower = faster, higher = more sensitive (default 9.5) |
| <code>max_seqs</code> | <code>int</code> | <code>1000</code> | Max prefilter targets per query. Raise to surface more hits at cost of runtime |
| <code>alignment_type</code> | <code>Literal[0, 1, 2, 3]</code> | <code>2</code> | Alignment scoring: 0=3Di SW, 1=TMalign, 2=3Di+AA (default), 3=LoL |
| <code>tmscore_threshold</code> | <code>float</code> | <code>0.0</code> | TM-score floor for hits (0-1). 0.0 keeps all; 0.5 ≈ same fold |
| <code>lddt_threshold</code> | <code>float</code> | <code>0.0</code> | LDDT floor for hits (0-1). 0.0 keeps all; 0.7+ = high local accuracy |
| <code>num_threads</code> | <code>int</code> | <code>4</code> | CPU threads for local search |
| <code>use_gpu</code> | <code>bool</code> | <code>False</code> | Local-only — run `--gpu 1` on a Linux x86_64 NVIDIA GPU host (driver >= 525.60.13). |

## Output API

In [4]:
display_api_reference("foldseek", "output", "run_foldseek_search")

**Output** — `FoldseekSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ticket_id</code> | <code>str</code> | required | Foldseek job ticket ID (remote only; empty in local mode) |
| <code>hits</code> | <code>list[FoldseekHit]</code> | <code>[]</code> | All alignment hits |
| <code>num_hits</code> | <code>int</code> | required | Total number of hits |
| <code>databases_queried</code> | <code>list[str]</code> | required | Databases included in the search |
| <code>result_url</code> | <code>str</code> | required | Foldseek result archive URL (remote only) |

**`FoldseekHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>database</code> | <code>str</code> | required | Source database the hit came from |
| <code>target_id</code> | <code>str</code> | required | Database-specific target identifier |
| <code>sequence_identity</code> | <code>float</code> | required | Sequence identity over aligned region (0-1) |
| <code>alignment_length</code> | <code>int</code> | required | Aligned region length in residues |
| <code>mismatches</code> | <code>int</code> | required | Mismatched columns |
| <code>gap_openings</code> | <code>int</code> | required | Gap-opening events |
| <code>query_start</code> | <code>int</code> | required | Query start (1-indexed) |
| <code>query_end</code> | <code>int</code> | required | Query end (1-indexed) |
| <code>target_start</code> | <code>int</code> | required | Target start (1-indexed) |
| <code>target_end</code> | <code>int</code> | required | Target end (1-indexed) |
| <code>evalue</code> | <code>float</code> | required | Foldseek e-value; smaller values are more significant |
| <code>bit_score</code> | <code>float</code> | required | Foldseek alignment bit score; higher is better |

## Step 1: fetch a query structure (TP53 from AFDB)

In [5]:
afdb = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(structure_format="pdb"),
)
print(f"Fetched {afdb.entry_id}: {afdb.sequence_length} residues, mean pLDDT={afdb.mean_plddt:.1f}")

Fetched AF-P04637-F1: 393 residues, mean pLDDT=75.1


## Step 2: search Foldseek (PDB100 only, ~30-60s)

In [6]:
output = run_foldseek_search(
    FoldseekSearchInput(structure=afdb.structure.structure_pdb),
    FoldseekSearchConfig(databases=["pdb100"]),
)
print(f"Foldseek returned {output.num_hits} hits across {output.databases_queried}")

Foldseek returned 890 hits across ['pdb100']


## Step 3: inspect top hits ranked by E-value

In [7]:
top_hits = sorted(output.hits, key=lambda h: h.evalue)[:10]
for hit in top_hits:
    print(
        f"{hit.target_id:<20} db={hit.database:<8} "
        f"evalue={hit.evalue:.2e}  identity={hit.sequence_identity:.1%}  "
        f"aligned={hit.alignment_length} aa"
    )

7t34-assembly1.cif.gz_A Crystal structure of AtDHDPR1 from Arabidopsis thaliana db=pdb100   evalue=2.90e-02  identity=8.5%  aligned=47 aa
4gkp-assembly1.cif.gz_A Structure of the truncated neck and C-terminal motor homology domain of ViK1 from Candida glabrata db=pdb100   evalue=3.40e-02  identity=4.9%  aligned=142 aa
5u5n-assembly1.cif.gz_B The dimeric crystal structure of HTPA Reductase from Sellaginella moellendorffii db=pdb100   evalue=3.40e-02  identity=8.1%  aligned=49 aa
3m7o-assembly3.cif.gz_C Crystal structure of mouse MD-1 in complex with phosphatidylcholine db=pdb100   evalue=4.50e-02  identity=9.1%  aligned=197 aa
5lgv-assembly1.cif.gz_B GlgE isoform 1 from Streptomyces coelicolor E423A mutant soaked in maltooctaose db=pdb100   evalue=4.50e-02  identity=6.3%  aligned=126 aa
5vt4-assembly2.cif.gz_C Sco GlgEI-V279S in complex with a pyrolidene-based methyl-phosphonate compound db=pdb100   evalue=5.10e-02  identity=6.3%  aligned=126 aa
4cn1-assembly1.cif.gz_B GlgE isoform 1 fr

## Cluster a candidate set of structures

`foldseek-cluster` groups a list of PDB structures by structural similarity. Local-only — runs the `foldseek easy-cluster` binary in a standalone subprocess.

In [8]:
cluster_output = run_foldseek_cluster(
    FoldseekClusterInput(
        structures=[afdb.structure.structure_pdb, afdb.structure.structure_pdb],
        structure_ids=["tp53_copy1", "tp53_copy2"],
    ),
    FoldseekClusterConfig(),
)
print(f"{cluster_output.num_clusters} cluster(s) over {cluster_output.num_structures} structure(s):")
for cluster in cluster_output.clusters:
    print(f"  {cluster.representative_id}: {cluster.member_ids}")

Running run_foldseek_cluster [00:00]

1 cluster(s) over 2 structure(s):
  tp53_copy2: ['tp53_copy2', 'tp53_copy1']


## Multimer (complex) search

`foldseek-multimer-search` takes a multi-chain PDB and searches against multimer-aware databases. Remote mode wraps the alignment mode as `complex-{base}` per the Foldseek server's wire protocol.

In [9]:
# Fetch a real multi-chain complex from RCSB: hemoglobin (4HHB), an alpha2-beta2 tetramer.
multi_output = run_foldseek_multimer_search(
    FoldseekMultimerSearchInput(structure=Structure.from_rcsb("4HHB")),
    FoldseekMultimerSearchConfig(databases=["pdb100"]),
)
print(f"Multimer search returned {multi_output.num_hits} hits")
for hit in sorted(multi_output.hits, key=lambda h: h.evalue)[:5]:
    print(f"  {hit.target_id}: evalue={hit.evalue:.2e}, identity={hit.sequence_identity:.1%}")

Multimer search returned 1912 hits
  1a00-assembly1.cif.gz_A HEMOGLOBIN (VAL BETA1 MET, TRP BETA37 TYR) MUTANT: evalue=1.00e+00, identity=100.0%
  1a01-assembly1.cif.gz_A HEMOGLOBIN (VAL BETA1 MET, TRP BETA37 ALA) MUTANT: evalue=1.00e+00, identity=100.0%
  1a0u-assembly1.cif.gz_C HEMOGLOBIN (VAL BETA1 MET) MUTANT: evalue=1.00e+00, identity=100.0%
  1a0z-assembly1.cif.gz_A HEMOGLOBIN (VAL BETA1 MET) MUTANT: evalue=1.00e+00, identity=100.0%
  3a0g-assembly1.cif.gz_A Crystal structure analysis of guinea pig oxyhemoglobin at 2.5 angstroms resolution: evalue=1.00e+00, identity=75.5%


## Multimer clustering

`foldseek-multimercluster` groups a set of multi-chain assemblies by combined chain + interface similarity. Local-only — clustering an arbitrary user set of complexes is not exposed by the public server.

In [10]:
# Cluster three real multimers from RCSB: PD-1/PD-L1 (3BIK, hetero-trimer)
# and two TIM dimers (1TIM, 8TIM). Expect two clusters: 3BIK alone, the two
# structurally homologous TIMs together.
multimer_ids = ["3BIK", "1TIM", "8TIM"]
mc_output = run_foldseek_multimercluster(
    FoldseekMultimerClusterInput(
        structures=[Structure.from_rcsb(pdb) for pdb in multimer_ids],
        structure_ids=multimer_ids,
    ),
    FoldseekMultimerClusterConfig(num_threads=2),
)
print(f"{mc_output.num_clusters} multimer clusters across {mc_output.num_multimers} input complexes")
for cluster in mc_output.clusters:
    print(f"  representative {cluster.representative_id}: {len(cluster.member_ids)} members ({cluster.member_ids})")

Running run_foldseek_multimercluster [00:00]

2 multimer clusters across 3 input complexes
  representative 3BIK: 1 members (['3BIK'])
  representative 8TIM: 2 members (['8TIM', '1TIM'])


## Reciprocal best hits

`foldseek-rbh` returns only mutual best hits between a query and a target DB — useful for structural orthology calls. Local-only (the public server does not expose RBH). Pass either a prebuilt Foldseek DB or a directory of PDBs (Foldseek auto-builds a temporary DB).

In [11]:
# Build a tiny target set from real PDBs and run RBH against it.
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    target_dir = Path(tmp)
    for pdb_id in ("1TIM", "8TIM"):
        (target_dir / f"{pdb_id}.pdb").write_text(Structure.from_rcsb(pdb_id).structure)

    rbh_output = run_foldseek_rbh(
        FoldseekRBHInput(structure=Structure.from_rcsb("1TIM")),
        FoldseekRBHConfig(local_db=str(target_dir), num_threads=2),
    )

print(f"RBH returned {rbh_output.num_hits} mutual best hits")
for hit in rbh_output.hits:
    print(f"  {hit.target_id}: evalue={hit.evalue:.2e}, identity={hit.sequence_identity:.1%}")

Running run_foldseek_rbh [00:00]

RBH returned 2 mutual best hits
  1TIM_A: evalue=2.37e-55, identity=100.0%
  1TIM_B: evalue=1.32e-54, identity=100.0%
